# ★ Mini-Capstone 2 — Clinical Triage Pipeline
**Goal:** Integrate Modules 1–5 into a production-style clinical triage system.  
**Time:** ~4 hours

> ⚠️ **Synthetic data only** — no real patient records are used.

---
### What you'll build
| Step | Skill | Deliverable |
|------|-------|-------------|
| 1 | M2 — Pydantic + Instructor | 100 TriageCard objects (≥ 95% pass rate) |
| 2 | M5 — XGBoost + embeddings | Triage classifier (AUC ≥ 0.68) |
| 3 | M6 preview — LLM-as-judge | 30 cards evaluated (avg ≥ 3.0/5.0) |
| 4 | M1 — Cost tracking | mini_capstone_2_report.md |


In [ ]:
import sys, os
sys.path.insert(0, '.')
from llm_checker import check, hint, evaluate, progress, show_exercise
print("✅ llm_checker loaded")


In [ ]:
import importlib
for pkg in ["openai","instructor","pydantic","sklearn","xgboost","sentence_transformers"]:
    print(f'{"✅" if importlib.util.find_spec(pkg) else "❌"} {pkg}')


---
## Step 1 — Patient Data Preparation (M2 + M5 skill)

In [ ]:
show_exercise(
    "MC2-1", "Generate 100 TriageCards with Pydantic + Instructor",
    """Generate 100 synthetic TriageCard objects.
Fields: patient_id (str), age (int 0-120), chief_complaint (str),
        vital_signs (VitalSigns model), triage_level (Literal[1..5]),
        diagnosis_hypothesis (str).
Pydantic validation pass_rate must be >= 95%.""",
    hints=[
        "Generate a note text first, then extract TriageCard via instructor.patch(client)",
        "VitalSigns: heart_rate, bp_systolic, temperature, spo2",
        "Literal[1,2,3,4,5] enforces valid triage levels at parse time",
    ],
    checks=[
        "100 TriageCard objects created",
        "Pydantic pass rate >= 95%",
        "triage_level is always an int in {1,2,3,4,5}",
    ],
)


In [ ]:
from openai import OpenAI
from pydantic import BaseModel, Field
from typing import Optional, Literal, List
import instructor

lm = instructor.patch(OpenAI(base_url="http://localhost:1234/v1", api_key="lm-studio"))
raw_client = OpenAI(base_url="http://localhost:1234/v1", api_key="lm-studio")

class VitalSigns(BaseModel):
    heart_rate:   int
    bp_systolic:  int
    temperature:  float
    spo2:         int

class TriageCard(BaseModel):
    patient_id:           str
    age:                  int = Field(ge=0, le=120)
    chief_complaint:      str
    vital_signs:          VitalSigns
    triage_level:         Literal[1, 2, 3, 4, 5]
    diagnosis_hypothesis: str


def generate_triage_card(patient_num: int) -> Optional[TriageCard]:
    """Generate one synthetic TriageCard."""
    urgency = "life-threatening emergency" if patient_num % 5 == 0 else "routine visit"
    # Step 1 — generate a realistic clinical note
    note_resp = raw_client.chat.completions.create(
        model="local-model",
        messages=[{"role": "user", "content":
            f"Write a 3-sentence synthetic ER note for patient P{patient_num:03d} "
            f"presenting with a {urgency}. Include vitals and a brief plan."}],
        max_tokens=150,
    )
    note = note_resp.choices[0].message.content

    # Step 2 — extract structured TriageCard
    try:
        return lm.chat.completions.create(
            model="local-model",
            messages=[{"role": "user", "content":
                f"Extract triage data from this note. Patient ID: P{patient_num:03d}\n{note}"}],
            response_model=TriageCard,
            max_retries=2,
        )
    except Exception:
        return None


cards, failures = [], 0
for i in range(100):
    card = generate_triage_card(i)
    if card: cards.append(card)
    else:    failures += 1

pass_rate_mc2 = len(cards) / 100
print(f"Generated {len(cards)}/100  pass_rate = {pass_rate_mc2:.2%}")


In [ ]:
check([
    (len(cards) >= 95,                                     f">= 95 cards (got {len(cards)})"),
    (pass_rate_mc2 >= 0.95,                                f"Pass rate >= 95% (got {pass_rate_mc2:.1%})"),
    (all(isinstance(c.triage_level, int) for c in cards),  "triage_level is always int"),
    (all(1 <= c.triage_level <= 5 for c in cards),         "triage_level in [1,5]"),
], exercise_id="MC2-1")


---
## Step 2 — Triage Prediction (M5 skill)

In [ ]:
show_exercise(
    "MC2-2", "XGBoost triage classifier with embedding features",
    """Binary label: urgent = triage_level in {1, 2}.
Features: tabular (age + 4 vitals) + chief_complaint embedding (384-d).
Train/test split 80/20. Report AUC on held-out test set. Target: AUC >= 0.68.""",
    hints=[
        "y = [1 if c.triage_level <= 2 else 0 for c in cards]",
        "SentenceTransformer('all-MiniLM-L6-v2').encode([c.chief_complaint for c in cards])",
        "np.hstack([X_tab, X_emb]) — combine tabular + embedding",
    ],
    checks=[
        "Feature matrix has >= 5 columns",
        "AUC >= 0.68 on held-out test set",
        "train split only used for fitting",
    ],
)


In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier

embedder = SentenceTransformer("all-MiniLM-L6-v2")

# TODO: build X_tab (age + vitals), X_emb (complaint embeddings), y
X_tab = None
X_emb = None
y_mc2 = None

# TODO: combine and split
X_mc2      = None   # np.hstack([X_tab, X_emb])
triage_auc = 0.0
triage_model = None


In [ ]:
check([
    (X_mc2 is not None,                  "Feature matrix built"),
    (X_mc2 is not None and X_mc2.shape[1] >= 5,  "At least 5 feature columns"),
    (triage_model is not None,           "XGBoost model trained"),
    (triage_auc >= 0.68,                 f"AUC >= 0.68 (got {triage_auc:.3f})"),
], exercise_id="MC2-2")


---
## Step 3 — LLM-as-Judge Evaluation (M6 preview)

In [ ]:
show_exercise(
    "MC2-3", "LLM-as-judge on 30 triage cards",
    """For 30 cards: show triage_level + diagnosis_hypothesis + model prediction to an LLM judge.
Score 1-5: does the diagnosis support the predicted urgency?
Average score must be >= 3.0/5.0.""",
    hints=[
        "Judge system: 'Score 1-5 (integer only). 1=not aligned, 5=fully aligned.'",
        "re.findall(r'\\d', response)[0] to parse the score",
        "If triage_model is None, use a heuristic prediction for the demo",
    ],
    checks=[
        "30 cards judged",
        "avg_judge_score >= 3.0",
    ],
)


In [ ]:
import re, statistics

JUDGE_SYS = (
    "You are a clinical quality reviewer. "
    "Score 1-5 (integer only). 1 = diagnosis completely unaligned with urgency, "
    "5 = diagnosis strongly supports the urgency assessment."
)

def judge_triage_card(card: TriageCard, predicted_urgent: bool) -> int:
    prompt = (
        f"Patient age {card.age}. Chief complaint: {card.chief_complaint}.\n"
        f"Diagnosis hypothesis: {card.diagnosis_hypothesis}\n"
        f"Triage level: {card.triage_level} (1=most urgent).  "
        f"Model predicted urgent: {predicted_urgent}.\n"
        "Score (1-5):"
    )
    resp = raw_client.chat.completions.create(
        model="local-model",
        messages=[{"role": "system", "content": JUDGE_SYS},
                  {"role": "user",   "content": prompt}],
        max_tokens=5, temperature=0,
    )
    nums = re.findall(r"\d", resp.choices[0].message.content)
    return int(nums[0]) if nums else 3


judge_scores_mc2 = []
for card in cards[:30]:
    if triage_model is not None and X_mc2 is not None:
        # real prediction
        idx = cards.index(card)
        pred_prob = float(triage_model.predict_proba(X_mc2[idx:idx+1])[0][1])
        predicted_urgent = pred_prob >= 0.5
    else:
        predicted_urgent = card.triage_level <= 2   # fallback heuristic

    score = judge_triage_card(card, predicted_urgent)
    judge_scores_mc2.append(score)

avg_judge_mc2 = statistics.mean(judge_scores_mc2) if judge_scores_mc2 else 0
print(f"Avg judge score: {avg_judge_mc2:.2f}/5.0")

check([
    (len(judge_scores_mc2) == 30, "30 cards judged"),
    (avg_judge_mc2 >= 3.0,        f"Avg score >= 3.0 (got {avg_judge_mc2:.2f})"),
], exercise_id="MC2-3")


---
## Step 4 — Cost Breakdown + Report (🔴 Challenge)

In [ ]:
show_exercise(
    "MC2-4", "Cost breakdown and final report",
    """Track all LLM calls: note generation (100), instructor extraction (100), judge (30).
Compute estimated cloud cost (gpt-4o-mini pricing: $0.15/1M input, $0.60/1M output).
Write mini_capstone_2_report.md with all metrics and cost comparison.""",
    hints=[
        "LM Studio cost = $0.00 per call (local)",
        "Rough token estimate: len(text.split()) * 1.3",
        "Cost = (input_tok * 0.15 + output_tok * 0.60) / 1_000_000",
    ],
    checks=[
        "Cost breakdown by step computed",
        "Hybrid vs cloud-only comparison in report",
        "mini_capstone_2_report.md written with all checklist items",
    ],
)


In [ ]:
# Estimated token usage (rough)
AVG_NOTE_TOKENS   = 200   # generation
AVG_EXTRACT_TOKENS= 300   # instructor extraction
AVG_JUDGE_TOKENS  = 100   # judge call

GPT_INPUT  = 0.15 / 1_000_000
GPT_OUTPUT = 0.60 / 1_000_000

step_costs = {
    "Note generation (100)":   100 * AVG_NOTE_TOKENS   * (GPT_INPUT + GPT_OUTPUT),
    "Extraction (100)":        100 * AVG_EXTRACT_TOKENS * (GPT_INPUT + GPT_OUTPUT),
    "LLM-as-judge (30)":        30 * AVG_JUDGE_TOKENS   * (GPT_INPUT + GPT_OUTPUT),
}
total_cloud_cost = sum(step_costs.values())
cost_rows = "\n".join(f"| {step} | $0.0000 | ${cost:.4f} |"
                       for step, cost in step_costs.items())

report_mc2 = f"""# Mini-Capstone 2 Report — Clinical Triage Pipeline

## Metrics

| Metric | Value | Target | OK |
|--------|-------|--------|----|
| TriageCards generated | {len(cards)}/100 | 100 | {"✅" if len(cards)>=95 else "❌"} |
| Pydantic pass rate | {pass_rate_mc2:.1%} | >= 95% | {"✅" if pass_rate_mc2>=0.95 else "❌"} |
| XGBoost AUC | {triage_auc:.3f} | >= 0.68 | {"✅" if triage_auc>=0.68 else "❌"} |
| LLM-as-judge avg | {avg_judge_mc2:.2f}/5.0 | >= 3.0 | {"✅" if avg_judge_mc2>=3.0 else "❌"} |

## Cost Comparison

| Step | LM Studio | GPT-4o-mini (est.) |
|------|-----------|---------------------|
{cost_rows}
| **Total** | **$0.0000** | **${total_cloud_cost:.4f}** |

## Checklist

- [{"x" if len(cards) >= 95 else " "}] 100 TriageCards in triage_cards.jsonl (all fields present)
- [{"x" if triage_auc >= 0.68 else " "}] XGBoost AUC >= 0.68 on held-out test set
- [{"x" if pass_rate_mc2 >= 0.95 else " "}] Pydantic validation pass rate >= 95%
- [{"x" if avg_judge_mc2 >= 3.0 else " "}] LLM-as-judge average score >= 3.0/5.0 (30 cards)
- [x] Cost breakdown by step computed
- [x] Hybrid vs cloud-only cost comparison included
- [x] Pipeline runs end-to-end without manual intervention
"""

with open("mini_capstone_2_report.md", "w") as f:
    f.write(report_mc2)
print("✅ mini_capstone_2_report.md written")
print(report_mc2[:600])


In [ ]:
check([
    (len(cards) >= 95,                                      f"TriageCards: {len(cards)}/100"),
    (pass_rate_mc2 >= 0.95,                                 f"Pass rate: {pass_rate_mc2:.1%}"),
    (triage_auc >= 0.68,                                    f"AUC: {triage_auc:.3f}"),
    (avg_judge_mc2 >= 3.0,                                  f"Judge: {avg_judge_mc2:.2f}"),
    (os.path.exists("mini_capstone_2_report.md"),            "Report written"),
], exercise_id="MC2-final")
progress()
